# NLF — run & time it (fold, no-fold, competitors)

[**NLF**](https://github.com/orenlivne/nlf) (Nonlinear Flow / "Flow Algebraic Multigrid") is a
nonlinear, box-constrained network-flow solver built **on top of** the LAMG+ graph-Laplacian
multigrid solver. This notebook **runs and times** the three headline results from the
[paper](../doc/nlf_sisc.pdf): (1) **max-flow exactness** at the fold `F*`, (2) the **inner-solver
swap** — the inner Laplacian solve is a replaceable module, so every converging inner solver returns
the *identical* `F*` — and (3) **no-fold congestion (BPR)**, where NLF races Ipopt and L-BFGS on the
same convex Beckmann program.

All numbers below are **executed**, not transcribed; the final cell prints them next to the paper's.

## Setup
Activate this notebook's environment and load the reproduction helpers (the fold loader, the four
inner-solver engines, and the no-fold BPR solver + competitors), extracted from the repo's benchmark
scripts.

> **One-time setup** (from the repo root):
> `julia --project=examples/repro_env -e 'using Pkg; Pkg.instantiate()'`, then register an IJulia
> kernel (`julia --project=examples/repro_env -e 'using Pkg; Pkg.add("IJulia"); using IJulia; installkernel("Julia")'`).

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "repro_env"); io = devnull)   # NLF + LAMG+ + JuMP/Ipopt + Optim + Laplacians
include(joinpath(@__DIR__, "repro_lib.jl"))
println("ready — Julia ", VERSION)

ready — Julia 1.12.6


## Max-flow exactness  (paper Table&nbsp;`tab:mf`)

NLF reaches the true max-flow `F*` (chord-Newton with pseudo-arclength continuation *through* the
fold). The reference `F*` is the paper's value (LP over free flows and push–relabel agree to
`1e-4`). We run the exact ground-truth NLF solve (`inner=:direct`) on the three DIMACS-classical
generators and check `NLF/F* ≈ 1.0000`.

In [2]:
mf_cases = [
    ("grid2d/16",  () -> nlf_grid2d(16; cap_lo=0.1, cap_hi=10.0, rng=MersenneTwister(3)),  4.973),
    ("grid3d/6",   () -> nlf_grid3d(6;  cap_lo=0.1, cap_hi=10.0, rng=MersenneTwister(8)),  17.478),
    ("washington", () -> nlf_washington(8, 8; rng=MersenneTwister(12)),                    500.745),
]
@printf("%-12s %5s  %-12s  %-12s  %-9s  %-6s\n", "graph","n","paper F*","NLF F*","NLF/F*","conv")
println("-"^64)
mf_rows = []
for (name, gen, paperF) in mf_cases
    mfp = gen(); n = size(mfp.A, 1)
    Fstar, φ, f, info = nlf_maxflow(mfp; tol=1e-8, inner=:direct)
    @printf("%-12s %5d  %-12.4f  %-12.4f  %-9.4f  %-6s\n", name, n, paperF, Fstar, Fstar/paperF, info.converged)
    push!(mf_rows, (name=name, paperF=paperF, nlfF=Fstar, ratio=Fstar/paperF))
end

graph            n  paper F*      NLF F*        NLF/F*     conv  
----------------------------------------------------------------


grid2d/16      256  4.9730        4.9734        1.0001     true  


grid3d/6       216  17.4780       17.4777       1.0000     true  
washington      66  500.7450      500.7448      1.0000     true  


## The fold: swap the inner solver  (paper Table&nbsp;`tab:innerfold`)

Near the max-flow fold the Jacobian `J = B·diag(ρ')·Bᵀ` is near-singular. The deflation in NLF's
outer continuation — *not* the inner solver — is what carries it through, so the inner Laplacian
solve is a replaceable module. We hold the outer method fixed and swap the inner solver across four
engines:

| engine | inner solve |
|---|---|
| **LAMG+** | algebraic multigrid (the package's default) |
| **approxChol** | approximate Cholesky (Laplacians.jl) |
| **direct** | exact sparse Cholesky |
| **diagPCG** | diagonally-preconditioned CG (no coarse correction) |

**Every converging solver returns the identical `F*`.** The operative distinction is near-linear vs.
not: `direct` is fastest where it fits, `approxChol`/`LAMG+` are the near-linear options, `diagPCG`
(no coarse grid) is the least scalable.

In [3]:
fold_cases = [("grid2","AG-Monien__grid2.mtx",3.003), ("circuit_1","Bomhof__circuit_1.mtx",0.944)]
# warm-up (JIT) all four engines once
let mfp = mtx_maxflow(data_path("AG-Monien__grid2.mtx"))
    for (_, ic) in INNERS; try; nlf_maxflow(mfp; inner=ic, max_steps=30); catch; end; end
end
fold_rows = []
for (label, fn, paperF) in fold_cases
    mfp = mtx_maxflow(data_path(fn)); n = size(mfp.B,1); m = size(mfp.B,2)
    println("\n$label  ($fn)   n=$n  m=$m   paper F*=$paperF")
    @printf("    %-11s  %-12s  %-7s  %-9s  %s\n", "inner","F*","steps","time_s","converged")
    println("    ", "-"^56)
    for (nm, ic) in INNERS
        t = @elapsed ((Fs,φ,f,info) = nlf_maxflow(mfp; inner=ic, max_steps=120, tol=1e-7))
        @printf("    %-11s  %-12.6f  %-7d  %-9.3f  %s\n", nm, Fs, info.steps, t, info.converged)
        push!(fold_rows, (label=label, inner=nm, Fstar=Fs, t=t, conv=info.converged, paperF=paperF))
    end
end

PCG Stopped by stagnation test 5


PCG Stopped by stagnation test 5
PCG Stopped by stagnation test 5


PCG Stopped by stagnation test 5
PCG Stopped by stagnation test 5
PCG Stopped by stagnation test 5
PCG Stopped by stagnation test 5
PCG Stopped by stagnation test 5
PCG Stopped by stagnation test 5


PCG Stopped by stagnation test 5

grid2  (AG-Monien__grid2.mtx)   n=3296  m=6432   paper F*=3.003


    inner        F*            steps    time_s     converged
    --------------------------------------------------------
    LAMG+        3.002798      29       11.343     true
PCG Stopped by stagnation test 

...


PCG Stopped by stagnation test 5
    approxChol   3.002798      29       0.866      true
    direct       3.002799      1        0.304      true


    diagPCG      3.002798      29       12.365     true

circuit_1  (Bomhof__circuit_1.mtx)   n=2624  m=16624   paper F*=0.944


    inner        F*            steps    time_s     converged
    --------------------------------------------------------
    LAMG+        0.944332      5        3.995      true


    approxChol   0.944332      5        1.110      true
    direct       0.944332      3        0.407      true


    diagPCG      0.944332      5        0.635      true


## No-fold congestion vs. the competitors

Congestion flow (BPR / Beckmann) has *no* fold — a handful of warm-started Newton steps converge.
On the same convex program we race three paradigms:

* **NLF** — multigrid-Newton (one LAMG+ hierarchy, lazily refreshed, shared across load steps),
* **Ipopt** — interior-point with a sparse-direct KKT solve,
* **L-BFGS-on-dual** — a matrix-free first-order method.

NLF converges in ~2–4 linear-solve-cost (a "few cycles on the nonlinear residual") and is the
fastest of the three on this circuit graph.

In [4]:
B0 = load_mtx(data_path("Bomhof__circuit_1.mtx"))
B, c, t0, b, p = instance(B0); n = size(B,1); m = size(B,2)
L = make_law(c, t0, b, p)
s, t = far_pair(B); d = zeros(n); d[s] = -1.0; d[t] = 1.0
α = 0.3 * median_(c)
println("graph = Bomhof__circuit_1 (LCC)   n=$n  m=$m   α=$(round(α; digits=4))")

# warm-up (JIT) each solver
nlf_bpr(B, L, d, α; inner=:multigrid); lbfgs_dual(B, L, d, α; tlim=5.0); ipopt_bpr(B, L, d, α; tlim=30.0)

φ, fF, st, r = nlf_bpr(B, L, d, α; inner=:multigrid, tlim=600.0)
tNLF = @elapsed nlf_bpr(B, L, d, α; inner=:multigrid, tlim=600.0)
fI, tIP, stIP = ipopt_bpr(B, L, d, α; tlim=60.0)
lb = lbfgs_dual(B, L, d, α; tlim=60.0)

@printf("\n%-16s  %-9s  %-7s  %-9s  %s\n", "solver","time_s","steps","resid","converged")
println("-"^58)
@printf("%-16s  %-9.4f  %-7d  %-9.2e  %s\n", "NLF (multigrid)", tNLF, st, r, r < 1e-6)
@printf("%-16s  %-9.4f  %-7s  %-9s  %s\n", "Ipopt (IPM)",    tIP, "-", "-", string(stIP))
@printf("%-16s  %-9.4f  %-7d  %-9.2e  %s\n", "L-BFGS (dual)",  lb.t, lb.it, lb.r, lb.c)
nofold = (tNLF=tNLF, r=r, tIP=tIP, stIP=string(stIP), tLB=lb.t, cLB=lb.c);

graph = Bomhof__circuit_1 (LCC)   n=2624  m=16624   α=0.5238



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************


solver            time_s     steps    resid      converged
----------------------------------------------------------


NLF (multigrid)   0.0231     20       1.38e-10   true


Ipopt (IPM)       0.0889     -        -          LOCALLY_SOLVED
L-BFGS (dual)     0.3949     498      2.93e-09   true


## Reproduced vs. paper

A final side-by-side. Every reproduced number matches the paper to the reported precision.

In [5]:
println("[tab:mf] max-flow exactness  (NLF/F* ≈ 1.0000):")
for r in mf_rows
    @printf("   %-12s paper F*=%-9.3f  NLF F*=%-9.4f  NLF/F*=%.4f\n", r.name, r.paperF, r.nlfF, r.ratio)
end
println("\n[tab:innerfold] every converging inner solver returns the SAME F*:")
for label in unique(getfield.(fold_rows, :label))
    rows = filter(x -> x.label == label, fold_rows); paperF = rows[1].paperF
    Fs = [r.Fstar for r in rows if r.conv]
    @printf("   %-10s paper F*=%-7.3f  reproduced F*=%.4f  (spread across solvers=%.1e)\n",
            label, paperF, Fs[1], maximum(Fs)-minimum(Fs))
    conv = sort(filter(x->x.conv, rows), by=x->x.t)
    @printf("       fastest→slowest: %s\n", join(["$(r.inner)=$(round(r.t;digits=2))s" for r in conv], "  "))
end
println("\n[no-fold] NLF vs competitors on Bomhof__circuit_1:")
@printf("   NLF=%.3fs (resid %.1e)   Ipopt=%.3fs (%s)   L-BFGS=%.3fs (conv=%s)\n",
        nofold.tNLF, nofold.r, nofold.tIP, nofold.stIP, nofold.tLB, nofold.cLB)

[tab:mf] max-flow exactness  (NLF/F* ≈ 1.0000):
   grid2d/16    paper F*=4.973      NLF F*=4.9734     NLF/F*=1.0001
   grid3d/6     paper F*=17.478     NLF F*=17.4777    NLF/F*=1.0000
   washington   paper F*=500.745    NLF F*=500.7448   NLF/F*=1.0000

[tab:innerfold] every converging inner solver returns the SAME F*:
   grid2      paper F*=3.003    reproduced F*=3.0028  (spread across solvers=4.8e-08)


       fastest→slowest: direct=0.3s  approxChol=0.87s  LAMG+=11.34s  diagPCG=12.36s
   circuit_1  paper F*=0.944    reproduced F*=0.9443  (spread across solvers=2.3e-12)
       fastest→slowest: direct=0.41s  diagPCG=0.63s  approxChol=1.11s  LAMG+=3.99s

[no-fold] NLF vs competitors on Bomhof__circuit_1:
   NLF=0.023s (resid 1.4e-10)   Ipopt=0.089s (LOCALLY_SOLVED)   L-BFGS=0.395s (conv=true)
